# Notebook 04 — Statistical Analysis
**Dataset:** UK Online Retail (carrie1/ecommerce-data)  
**Scope:** RFM Segmentation · Hypothesis Testing · Time Series Forecasting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, f_oneway, chi2_contingency
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load Cleaned Data

In [ ]:
# Load from processed folder (or Colab path if running in Colab)
import os
data_path = '../data/processed/cleaned_data.csv'
if not os.path.exists(data_path):
    data_path = '../Project/cleaned_data.csv'
if not os.path.exists(data_path):
    data_path = 'cleaned_data.csv'

df = pd.read_csv(data_path, encoding='latin1')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID'] = df['CustomerID'].astype(int)
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Remove Dec 2011 (partial month — only 9 days)
df = df[~((df['InvoiceDate'].dt.year == 2011) & (df['InvoiceDate'].dt.month == 12))]

print(f'Shape: {df.shape}')
print(f'Date range: {df["InvoiceDate"].min()} → {df["InvoiceDate"].max()}')
print(f'Unique customers: {df["CustomerID"].nunique()}')
df.head(3)

## 2. RFM Analysis — Customer Segmentation

In [ ]:
# Reference date = 1 day after last invoice
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('Revenue',     'sum')
).reset_index()

print('RFM summary:')
print(rfm.describe().round(2))

In [ ]:
# Score each dimension 1–5 (5 = best)
rfm['R_Score'] = pd.qcut(rfm['Recency'],   5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  5, labels=[1,2,3,4,5]).astype(int)

rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# Segment labels
def segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Recent Customers'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Lost'
    else:
        return 'Potential Loyalists'

rfm['Segment'] = rfm.apply(segment, axis=1)
print(rfm['Segment'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Segment distribution
seg_counts = rfm['Segment'].value_counts()
axes[0].barh(seg_counts.index, seg_counts.values, color=sns.color_palette('muted', len(seg_counts)))
axes[0].set_title('Customer Segments (RFM)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Customers')

# Average Monetary by Segment
seg_monetary = rfm.groupby('Segment')['Monetary'].mean().sort_values(ascending=False)
axes[1].bar(seg_monetary.index, seg_monetary.values, color=sns.color_palette('Set2', len(seg_monetary)))
axes[1].set_title('Avg Revenue by Segment', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Avg Revenue (£)')
axes[1].tick_params(axis='x', rotation=30)

# RFM Total score distribution
axes[2].hist(rfm['RFM_Total'], bins=12, color='steelblue', edgecolor='white')
axes[2].set_title('RFM Total Score Distribution', fontsize=13, fontweight='bold')
axes[2].set_xlabel('RFM Total Score')
axes[2].set_ylabel('Customers')

plt.tight_layout()
plt.savefig('../tableau/screenshots/rfm_segmentation.png', bbox_inches='tight')
plt.show()
print('Saved rfm_segmentation.png')

## 3. Hypothesis Testing

### H1: UK customers have significantly higher Average Order Value than non-UK customers

In [ ]:
aov = df.groupby(['InvoiceNo', 'Country'])['Revenue'].sum().reset_index()
aov['is_uk'] = aov['Country'] == 'United Kingdom'

uk_aov     = aov[aov['is_uk']]['Revenue']
non_uk_aov = aov[~aov['is_uk']]['Revenue']

t_stat, p_val = ttest_ind(uk_aov, non_uk_aov, equal_var=False)  # Welch t-test

print('H1: UK vs Non-UK Average Order Value')
print(f'  UK     mean AOV: £{uk_aov.mean():.2f}  (n={len(uk_aov):,})')
print(f'  Non-UK mean AOV: £{non_uk_aov.mean():.2f}  (n={len(non_uk_aov):,})')
print(f'  t-statistic: {t_stat:.4f}')
print(f'  p-value:     {p_val:.6f}')
print(f'  Result: {"REJECT H0 — significant difference" if p_val < 0.05 else "FAIL TO REJECT H0"} (α=0.05)')

### H2: Revenue differs significantly across quarters (Q1–Q4 2011)

In [ ]:
df_2011 = df[df['InvoiceDate'].dt.year == 2011].copy()
df_2011['Quarter'] = df_2011['InvoiceDate'].dt.quarter

quarterly_rev = df_2011.groupby(['InvoiceNo', 'Quarter'])['Revenue'].sum().reset_index()
groups = [quarterly_rev[quarterly_rev['Quarter'] == q]['Revenue'].values for q in [1, 2, 3, 4]]

f_stat, p_val = f_oneway(*groups)

print('H2: Revenue differs across Q1–Q4 2011 (One-Way ANOVA)')
for q, g in zip([1,2,3,4], groups):
    print(f'  Q{q}: mean=£{np.mean(g):.2f}, n={len(g):,}')
print(f'  F-statistic: {f_stat:.4f}')
print(f'  p-value:     {p_val:.6f}')
print(f'  Result: {"REJECT H0 — quarterly revenue differs significantly" if p_val < 0.05 else "FAIL TO REJECT H0"} (α=0.05)')

### H3: Purchase volume differs across days of the week

In [ ]:
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek  # 0=Mon, 6=Sun
day_invoice = df.groupby(['InvoiceNo', 'DayOfWeek'])['Revenue'].sum().reset_index()

day_groups = [day_invoice[day_invoice['DayOfWeek'] == d]['Revenue'].values for d in range(7)]
day_names  = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

# Remove days with no data
valid = [(n, g) for n, g in zip(day_names, day_groups) if len(g) > 0]
valid_names, valid_groups = zip(*valid)

f_stat, p_val = f_oneway(*valid_groups)

print('H3: Revenue differs across days of the week (One-Way ANOVA)')
for n, g in zip(valid_names, valid_groups):
    print(f'  {n}: mean=£{np.mean(g):.2f}, n={len(g):,}')
print(f'  F-statistic: {f_stat:.4f}')
print(f'  p-value:     {p_val:.6f}')
print(f'  Result: {"REJECT H0 — day of week has significant effect" if p_val < 0.05 else "FAIL TO REJECT H0"} (α=0.05)')

## 4. Time Series Analysis & Forecasting

In [ ]:
# Monthly revenue — exclude partial Dec 2011 (already removed)
monthly = df.groupby(df['InvoiceDate'].dt.to_period('M'))['Revenue'].sum().reset_index()
monthly.columns = ['Period', 'Revenue']
monthly['Period_dt'] = monthly['Period'].dt.to_timestamp()
monthly = monthly.sort_values('Period_dt').reset_index(drop=True)

print('Monthly Revenue:')
print(monthly[['Period', 'Revenue']].to_string(index=False))

In [ ]:
from scipy.stats import pearsonr

# Simple linear trend regression
x = np.arange(len(monthly))
y = monthly['Revenue'].values
slope, intercept, r_value, p_val, std_err = stats.linregress(x, y)

trend_line = slope * x + intercept

# Forecast next 3 months
future_x    = np.arange(len(monthly), len(monthly) + 3)
future_rev  = slope * future_x + intercept
future_months = pd.date_range(monthly['Period_dt'].iloc[-1] + pd.DateOffset(months=1), periods=3, freq='MS')

print(f'Trend: slope=£{slope:,.0f}/month, R²={r_value**2:.3f}, p={p_val:.4f}')
print(f'\nForecast (Linear Extrapolation):')
for m, r in zip(future_months, future_rev):
    print(f'  {m.strftime("%Y-%m")}: £{r:,.0f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Actual + trend + forecast
axes[0].plot(monthly['Period_dt'], monthly['Revenue']/1e3, 'o-', color='steelblue', label='Actual Revenue', linewidth=2)
axes[0].plot(monthly['Period_dt'], trend_line/1e3, '--', color='orange', label='Linear Trend', linewidth=1.5)
axes[0].plot(future_months, future_rev/1e3, 's--', color='red', label='Forecast (3 months)', linewidth=1.5)
axes[0].set_title('Monthly Revenue — Actual vs Trend vs Forecast', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Revenue (£ thousands)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

# Plot 2: Month-over-month growth rate
monthly['MoM_Growth'] = monthly['Revenue'].pct_change() * 100
colors = ['green' if v >= 0 else 'red' for v in monthly['MoM_Growth'].fillna(0)]
axes[1].bar(monthly['Period_dt'], monthly['MoM_Growth'].fillna(0), color=colors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Month-over-Month Revenue Growth (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Growth (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../tableau/screenshots/time_series_forecast.png', bbox_inches='tight')
plt.show()

## 5. Customer Cohort Analysis

In [ ]:
# Assign each customer to their first-purchase cohort month
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')
cohort_map = df.groupby('CustomerID')['InvoiceMonth'].min().rename('CohortMonth')
df = df.join(cohort_map, on='CustomerID')
df['CohortIndex'] = (df['InvoiceMonth'].dt.year  - df['CohortMonth'].dt.year) * 12 + \
                    (df['InvoiceMonth'].dt.month - df['CohortMonth'].dt.month)

cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['CustomerID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='CustomerID')

cohort_size = cohort_pivot.iloc[:, 0]
retention   = cohort_pivot.divide(cohort_size, axis=0)

print('Cohort Retention Matrix (first 6 months):')
print(retention.iloc[:, :6].round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
retention_display = retention.iloc[:, :10].copy()
retention_display.index = retention_display.index.astype(str)

sns.heatmap(
    retention_display,
    annot=True, fmt='.0%',
    cmap='YlGnBu',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Retention Rate'}
)
ax.set_title('Customer Cohort Retention Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Months After First Purchase')
ax.set_ylabel('Cohort Month')
plt.tight_layout()
plt.savefig('../tableau/screenshots/cohort_retention.png', bbox_inches='tight')
plt.show()

## 6. Summary — Key Statistical Findings

In [ ]:
summary = {
    'Champions':         rfm[rfm['Segment']=='Champions'].shape[0],
    'At Risk':           rfm[rfm['Segment']=='At Risk'].shape[0],
    'Lost':              rfm[rfm['Segment']=='Lost'].shape[0],
    'UK mean AOV':       f'£{uk_aov.mean():.2f}',
    'Non-UK mean AOV':   f'£{non_uk_aov.mean():.2f}',
    'H1 p-value':        round(p_val, 6),
    'Monthly revenue trend': f'£{slope:,.0f}/month (upward)',
}

# Export RFM for Tableau
rfm.to_csv('../data/processed/rfm_segments.csv', index=False)
monthly[['Period', 'Revenue', 'MoM_Growth']].to_csv('../data/processed/monthly_revenue.csv', index=False)

print('Statistical Analysis Complete.')
print('Exports: rfm_segments.csv, monthly_revenue.csv → data/processed/')
for k, v in summary.items():
    print(f'  {k}: {v}')